<a href="https://colab.research.google.com/github/Vishweshwar001/Dendraite.ai-Task/blob/main/dendrite.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Dendrite.ai Assignment**

# Import Libraries

In [34]:
import pandas as pd
import numpy as np
import json
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.decomposition import PCA
from sklearn.model_selection import GridSearchCV, train_test_split, KFold
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.feature_selection import SelectFromModel
from sklearn.compose import ColumnTransformer
from sklearn.base import BaseEstimator, TransformerMixin
import warnings
warnings.filterwarnings('ignore')

# 1. Load JSON configuration


In [35]:
with open('algoparams_from_ui.json', 'r') as f:
    config = json.load(f)

settings = config['design_state_data']

# 2. Load Dataset


In [36]:
df = pd.read_csv('iris.csv')


# 3. Extract Target and Features


In [37]:
target_col = settings['target']['target']
prediction_type = settings['target']['prediction_type']

feature_handling = settings['feature_handling']
selected_features = [feat for feat, val in feature_handling.items() if val['is_selected']]

X = df[selected_features].copy()
y = df[target_col].copy()

# 4. Handle Missing Values


In [38]:
def create_imputer_pipeline(feature_handling):
    transformers = []
    for feature_name, params in feature_handling.items():
        if params['is_selected']:
            feat_type = params['feature_variable_type']
            details = params['feature_details']

            if feat_type == 'numerical':
                if details['missing_values'] == 'Impute':
                    if details['impute_with'] == 'Average of values':
                        strategy = 'mean'
                        imputer = SimpleImputer(strategy=strategy)
                    elif details['impute_with'] == 'custom':
                        fill_value = details['impute_value']
                        imputer = SimpleImputer(strategy='constant', fill_value=fill_value)
                    transformers.append((feature_name, imputer, [feature_name]))
            elif feat_type in ['text', 'categorical']:
                encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
                transformers.append((feature_name, encoder, [feature_name]))

    return ColumnTransformer(transformers)


# 5. Feature Engineering


In [39]:
def feature_engineering(X_df):
    X_df = pd.DataFrame(X_df)


    # Linear interactions
    if "petal_length" in X_df.columns and "sepal_width" in X_df.columns:
        X_df["petal_length*sepal_width"] = X_df["petal_length"] * X_df["sepal_width"]

    # Polynomial interactions
    if "petal_length" in X_df.columns and "sepal_width" in X_df.columns:
        X_df["petal_length/sepal_width"] = X_df["petal_length"] / (X_df["sepal_width"] + 1e-5)

    # Explicit pairwise interactions
    if "petal_width" in X_df.columns and "sepal_length" in X_df.columns:
        X_df["petal_width/sepal_length"] = X_df["petal_width"] / (X_df["sepal_length"] + 1e-5)

    return X_df

feature_gen_transformer = FunctionTransformer(feature_engineering)

# 6. Feature Reduction Methods


In [40]:
class CorrelationFeatureSelector(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=0.1):
        self.threshold = threshold

    def fit(self, X, y):
        corrs = pd.DataFrame(X).apply(lambda col: abs(np.corrcoef(col, y)[0, 1]))
        self.selected_features_ = corrs[corrs >= self.threshold].index.tolist()
        return self

    def transform(self, X):
        return pd.DataFrame(X).iloc[:, self.selected_features_]

feature_reduction_method = settings['feature_reduction']['feature_reduction_method']
num_features_to_keep = int(settings['feature_reduction']['num_of_features_to_keep'])

if feature_reduction_method == 'Tree-based':
    feature_selector = SelectFromModel(RandomForestRegressor(n_estimators=5, max_depth=6), max_features=num_features_to_keep)
elif feature_reduction_method == 'PCA':
    feature_selector = PCA(n_components=num_features_to_keep)
elif feature_reduction_method == 'Corr with Target':
    feature_selector = CorrelationFeatureSelector(threshold=0.1)
else:
    feature_selector = 'passthrough'


# 7. Model Factory Function


In [41]:
def get_model_and_params(model_key, model_config):
    if model_key == 'RandomForestRegressor':
        model = RandomForestRegressor()
        param_grid = {
            'model__n_estimators': [model_config['min_trees'], model_config['max_trees']],
            'model__max_depth': [model_config['min_depth'], model_config['max_depth']],
            'model__min_samples_leaf': [model_config['min_samples_per_leaf_min_value'], model_config['min_samples_per_leaf_max_value']]
        }
    elif model_key == 'LinearRegression':
        model = LinearRegression()
        param_grid = {}
    elif model_key == 'RidgeRegression':
        model = Ridge()
        param_grid = {
            'model__alpha': [model_config['min_regparam'], model_config['max_regparam']]
        }
    elif model_key == 'LassoRegression':
        model = Lasso()
        param_grid = {
            'model__alpha': [model_config['min_regparam'], model_config['max_regparam']]
        }
    elif model_key == 'ElasticNetRegression':
        model = ElasticNet()
        param_grid = {
            'model__alpha': [model_config['min_regparam'], model_config['max_regparam']],
            'model__l1_ratio': [model_config['min_elasticnet'], model_config['max_elasticnet']]
        }
    elif model_key == 'GBTRegressor':
        model = GradientBoostingRegressor()
        param_grid = {
            'model__n_estimators': model_config['num_of_BoostingStages'],
            'model__max_depth': [model_config['min_depth'], model_config['max_depth']],
            'model__learning_rate': [model_config['min_stepsize'], model_config['max_stepsize']]
        }
    elif model_key == 'DecisionTreeRegressor':
        model = DecisionTreeRegressor()
        param_grid = {
            'model__max_depth': [model_config['min_depth'], model_config['max_depth']],
            'model__min_samples_leaf': model_config['min_samples_per_leaf']
        }
    else:
        return None, None
    return model, param_grid

# 8. Run all selected models


In [42]:
algorithms = settings['algorithms']
selected_models = {k: v for k, v in algorithms.items() if v['is_selected'] and prediction_type == 'Regression'}


for model_key, model_config in selected_models.items():
    model, param_grid = get_model_and_params(model_key, model_config)
    if model is None:
        print(f"Skipping unsupported model: {model_key}")
        continue

    pipeline = Pipeline([
        ('imputer', create_imputer_pipeline(feature_handling)),
        ('feature_gen', feature_gen_transformer),
        ('feature_reduction', feature_selector),
        ('model', model)
    ])

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    cv = KFold(n_splits=6, shuffle=True, random_state=1)

    grid_search = GridSearchCV(pipeline, param_grid, cv=cv, scoring='r2', verbose=1)
    grid_search.fit(X_train, y_train)

    y_pred = grid_search.predict(X_test)
    r2 = r2_score(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))

    print(f"\nModel: {model_key}")
    print("Best Parameters:", grid_search.best_params_)
    print("R2 Score:", r2)
    print("RMSE:", rmse)

Fitting 6 folds for each of 8 candidates, totalling 48 fits

Model: RandomForestRegressor
Best Parameters: {'model__max_depth': 20, 'model__min_samples_leaf': 5, 'model__n_estimators': 20}
R2 Score: 0.9975561559240822
RMSE: 0.03941374206756656
